# 🧠 DocuMind v3 — Multimodal Agentic RAG Chat Assistant
### 100% free · open-source models · runs on Google Colab's free T4 GPU · no API keys

Chat with your PDFs, **including their figures and tables**. It remembers the conversation, so follow-ups like *"how many of them did they use?"* work, and you can attach your own images.

| Step | What happens | Industry name |
|---|---|---|
| 1 | Figures/tables in PDFs are rendered and described by a vision-language model | Multimodal ingestion |
| 2 | Follow-up questions are rewritten using chat history | Conversational memory / query condensation |
| 3 | Small talk vs document questions are separated | Intent routing |
| 4 | Question broken into search queries | Query planning |
| 5 | Keyword (BM25) + vector search, fused | Hybrid search + RRF |
| 6 | Cross-encoder re-scores results | Reranking |
| 7 | Each passage checked yes/no; refuse if nothing fits | Corrective RAG |
| 8 | Answer written with `[1]` citations, *looking at the figure image too* | Multimodal grounded generation |
| 9 | Answer fact-checked, rewritten if unsupported | Self-verification |
| 10 | Benchmark: 4 retrievers, 3 pipelines, multi-hop & refusal tests | Evaluation, LLM-as-judge |

---
### ▶️ How to run on Colab
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**

### ▶️ How to run on Kaggle (if Colab is down or out of GPU)
1. kaggle.com → **Create → New Notebook → File → Import Notebook** → upload this file
2. Right sidebar → **Session options**: Accelerator **GPU T4 x2**, Internet **On** (needs a one-time phone verification)
3. **Run All**. Results and charts appear under **Output** (`/kaggle/working`).
3. First run ≈ 10 min (model download + describing figures). Everything is stored on *Colab's* disk, not your computer.
4. Open the `gradio.live` link printed in section 8 and start chatting.

## 1 · Install libraries (on Colab's machine, not yours)

In [ ]:
!pip install -q rank_bm25 faiss-cpu pymupdf python-docx python-pptx 2>&1 | tail -n 2
# -U makes sure the libraries are new enough for Qwen2.5-VL (Kaggle's preinstalled ones can be older than Colab's)
!pip install -q -U transformers accelerate sentence-transformers gradio 2>&1 | tail -n 2
print("✅ Installed")

## 2 · Configuration

In [ ]:
import os, re, json, time, random, urllib.request
import numpy as np, pandas as pd, torch
from PIL import Image

GPU = torch.cuda.is_available()
if not GPU:
    raise RuntimeError("No GPU found. Colab: Runtime → Change runtime type → T4 GPU. Kaggle: Session options → Accelerator → GPU T4 x2. Then run all again.")
print("GPU:", torch.cuda.get_device_name(0))

# Open-source models (all free, downloaded from Hugging Face)
LLM_ID    = "Qwen/Qwen2.5-VL-3B-Instruct"            # ONE model that reads text AND images
EMBED_ID  = "BAAI/bge-small-en-v1.5"                  # turns text into vectors
RERANK_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"    # re-scores search results

CHUNK_SIZE, CHUNK_OVERLAP = 900, 150   # characters per chunk
TOP_K = 5                              # passages given to the LLM
RERANK_CONFIDENT = 2.0                 # trust a passage above this reranker score even if the LLM grader says no
MAX_FIGURE_PAGES = 16                  # pages per PDF to describe with the vision model (each takes ~10 s)
MAX_OCR_PAGES = 20                     # scanned pages per PDF to read with the vision model
HISTORY_TURNS = 3                      # past Q&A turns the model sees

# Chat memory is saved here so it survives re-running cells.
MEMORY_PATH = "chat_memory.json"
# To keep memory even after Colab disconnects, save it to your Google Drive (free):
# from google.colab import drive; drive.mount("/content/drive")
# MEMORY_PATH = "/content/drive/MyDrive/documind_chat_memory.json"

os.makedirs("docs", exist_ok=True); os.makedirs("figures", exist_ok=True)
random.seed(42)

## 3 · Load the models

In [ ]:
import transformers
from packaging import version
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText as VLM
except ImportError:
    from transformers import Qwen2_5_VLForConditionalGeneration as VLM
from sentence_transformers import SentenceTransformer, CrossEncoder

t0 = time.time()
dtype_kw = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"

# max_pixels caps how many tokens one image costs (keeps memory and speed in check on a T4)
processor = AutoProcessor.from_pretrained(LLM_ID, min_pixels=256 * 28 * 28, max_pixels=1024 * 28 * 28)

def load_llm(dtype):
    m = VLM.from_pretrained(LLM_ID, device_map={"": 0}, **{dtype_kw: dtype})  # one GPU (Kaggle gives 2 T4s; one is plenty)
    return m.eval()

model = load_llm(torch.float16)
embedder = SentenceTransformer(EMBED_ID, device="cuda")
reranker = CrossEncoder(RERANK_ID, device="cuda")
print(f"✅ Models loaded in {time.time()-t0:.0f}s · GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB of ~15 GB")

In [ ]:
@torch.no_grad()
def llm(prompt, system="You are a precise, helpful assistant.", max_new_tokens=400, image=None, history=None):
    """Ask the model. Optional: a PIL image, and past chat turns [{'role':..., 'content':...}]."""
    msgs = [{"role": "system", "content": [{"type": "text", "text": system}]}]
    for m in history or []:
        msgs.append({"role": m["role"], "content": [{"type": "text", "text": m["content"]}]})
    user = ([{"type": "image"}] if image is not None else []) + [{"type": "text", "text": prompt}]
    msgs.append({"role": "user", "content": user})
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image] if image is not None else None,
                       return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

def parse_json(text, default):
    """Small models sometimes wrap JSON in extra text — pull out the first JSON object/list."""
    m = re.search(r"\{.*\}|\[.*\]", text, re.S)
    if not m:
        return default
    try:
        return json.loads(m.group(0))
    except Exception:
        return default

# Safety check: some GPU/library combos make vision models output garbage in float16.
def looks_broken(s):
    return len(s.strip()) < 3 or s.count("!") > len(s) * 0.3
test = llm("In one sentence, what is retrieval-augmented generation?")
if looks_broken(test):
    print("⚠️ float16 output looks broken → reloading in bfloat16 (slower but stable)")
    del model; torch.cuda.empty_cache()
    model = load_llm(torch.bfloat16)
    test = llm("In one sentence, what is retrieval-augmented generation?")
print(test)

## 4 · Load documents — text **and** figures
- Text is extracted page by page and split into overlapping chunks.
- Pages with a figure/table caption (or a large embedded image) are rendered to PNG and **described by the vision model**. Those descriptions become searchable chunks that remember their image, so the answer can show and *look at* the figure.

Default documents: *Attention Is All You Need* and the original *RAG* paper. Upload your own in the app (section 8).

In [ ]:
import fitz  # PyMuPDF
import hashlib

SAMPLES = {
    "attention_is_all_you_need.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_lewis_2020.pdf": "https://arxiv.org/pdf/2005.11401",
}
for fn, url in SAMPLES.items():
    path = f"docs/{fn}"
    if not os.path.exists(path):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as r, open(path, "wb") as f:
                f.write(r.read())
        except Exception as e:
            print("⚠️ Could not download", fn, "-", e)

FIGURE_PROMPT = """This is a page from a PDF document. Describe every figure, diagram, chart and table on this page in detail:
what it shows, its components and labels, axes, important numbers, and the main takeaway.
Start each description with its label (e.g. "Figure 1:" or "Table 2:"). Ignore the ordinary body text."""
OCR_PROMPT = """This is a scanned or photographed page. Transcribe ALL the text on it exactly, in reading order.
Then, if there are any figures, diagrams, charts or tables, describe each one briefly after the heading "VISUALS:"."""
IMAGE_DOC_PROMPT = """Transcribe any text in this image, then describe the image in detail:
what it shows, labels, numbers, and the main takeaway."""
# Captions in many styles: "Figure 1:", "Fig 2 -", "Fig. 3.", "Table 1)", "Chart 4:", "Diagram 2 –" …
CAPTION_RE = re.compile(r"(?mi)^\s*((?:figure|fig\.?|table|chart|diagram|graph|exhibit|image)\s*\d+[a-z]?\s*[:.\-–—)|].{0,150})")

def visual_score(page):
    """How much of the page is pictures or vector drawings (charts/diagrams/tables exported from Office, Python, etc.)."""
    area = page.rect.width * page.rect.height or 1
    img_area = 0.0
    for info in page.get_image_info():
        x0, y0, x1, y1 = info["bbox"]
        img_area += max(0, x1 - x0) * max(0, y1 - y0)
    try:
        drawings = page.get_drawings()
    except Exception:
        drawings = []
    draw_area = 0.0
    if len(drawings) >= 20:  # a few lines = underline/border; many paths = chart, diagram or table
        box = fitz.Rect(drawings[0]["rect"])
        for d in drawings[1:]:
            box.include_rect(d["rect"])
        draw_area = box.width * box.height
    return min(1.0, img_area / area), min(1.0, draw_area / area)

def has_visual(captions, img_frac, draw_frac):
    return bool(captions) or img_frac > 0.05 or draw_frac > 0.10

FIG_CACHE_PATH = "figure_cache.json"
FIG_CACHE = json.load(open(FIG_CACHE_PATH)) if os.path.exists(FIG_CACHE_PATH) else {}

def file_key(path):
    """Short fingerprint of the file's contents, so two different files named 'notes.pdf' never share figures."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()[:8]

def chunk_text(name, page, text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Overlapping chunks that try to end on a sentence boundary."""
    text = re.sub(r"\s+", " ", text).strip()
    out, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            cut = text.rfind(". ", start + size // 2, end)
            if cut != -1:
                end = cut + 1
        out.append({"source": name, "page": page, "text": text[start:end].strip(), "kind": "text", "image": None})
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return out

def vision(img_path, prompt, max_new_tokens=450):
    """Run the vision model on an image file, cached so re-indexing is instant."""
    key = f"{img_path}::{prompt[:20]}"
    if key not in FIG_CACHE:
        FIG_CACHE[key] = llm(prompt, image=Image.open(img_path).convert("RGB"), max_new_tokens=max_new_tokens)
        json.dump(FIG_CACHE, open(FIG_CACHE_PATH, "w"))
    return FIG_CACHE[key]

def render_page(doc, pno, img_path):
    if not os.path.exists(img_path):
        doc[pno - 1].get_pixmap(matrix=fitz.Matrix(1.6, 1.6)).save(img_path)
    return img_path

def load_pdf(path, describe_figures=True, say=print):
    name = os.path.basename(path)
    stem = f"{re.sub(r'[^A-Za-z0-9_-]', '_', os.path.splitext(name)[0])[:40]}_{file_key(path)}"
    doc = fitz.open(path)
    chunks, fig_pages, scanned = [], [], []
    for pno, page in enumerate(doc, 1):
        text = page.get_text()
        if len(text.strip()) >= 50:
            chunks += chunk_text(name, pno, text)
            captions = [c.strip() for c in CAPTION_RE.findall(text)]
            img_frac, draw_frac = visual_score(page)
            if has_visual(captions, img_frac, draw_frac):
                fig_pages.append((pno, captions, img_frac + draw_frac))
        else:
            scanned.append(pno)  # no text layer → scanned / photographed page

    if scanned:  # read scanned pages with the vision model (free OCR)
        todo = scanned[:MAX_OCR_PAGES]
        for n, pno in enumerate(todo, 1):
            say(f"🔎 {name}: reading scanned page {pno} ({n}/{len(todo)})")
            img_path = render_page(doc, pno, f"figures/{stem}_p{pno}.png")
            text = vision(img_path, OCR_PROMPT, max_new_tokens=900)
            for c in chunk_text(name, pno, text):
                c["image"] = img_path  # keep the page image so answers can show it
                chunks.append(c)
        if len(scanned) > MAX_OCR_PAGES:
            say(f"⚠️ {name}: only the first {MAX_OCR_PAGES} of {len(scanned)} scanned pages were read (raise MAX_OCR_PAGES)")

    if describe_figures and fig_pages:
        # too many candidates? keep pages with captions first, then the most visual ones; process in page order
        best = sorted(fig_pages, key=lambda f: (bool(f[1]), f[2]), reverse=True)[:MAX_FIGURE_PAGES]
        todo = sorted(best, key=lambda f: f[0])
        say(f"📄 {name}: {len(fig_pages)} page(s) with figures/tables found, describing {len(todo)}")
        for n, (pno, captions, _) in enumerate(todo, 1):
            say(f"📷 {name}: describing figures on page {pno} ({n}/{len(todo)})")
            img_path = render_page(doc, pno, f"figures/{stem}_p{pno}.png")
            desc = vision(img_path, FIGURE_PROMPT)
            cap = " | ".join(captions) if captions else "image on page"
            chunks.append({"source": name, "page": pno, "kind": "figure", "image": img_path,
                           "text": f"[Figure/table on page {pno} of {name}] Captions: {cap}. Visual description: {desc}"})
    for c in chunks:
        c["path"] = path  # lets the app render a page preview later
    return chunks

def page_preview(chunk):
    """An image for a chunk: its figure, or a freshly rendered picture of its PDF page."""
    if chunk.get("image"):
        return chunk["image"]
    path = chunk.get("path")
    if not path or not path.lower().endswith(".pdf") or not os.path.exists(path):
        return None
    out = f"figures/preview_{file_key(path)}_p{chunk['page']}.png"
    if not os.path.exists(out):
        with fitz.open(path) as d:
            d[chunk["page"] - 1].get_pixmap(matrix=fitz.Matrix(1.4, 1.4)).save(out)
    return out

def load_docx(path):
    import docx
    d = docx.Document(path)
    parts = [p.text for p in d.paragraphs if p.text.strip()]
    for t in d.tables:
        for row in t.rows:
            parts.append(" | ".join(cell.text.strip() for cell in row.cells))
    return chunk_text(os.path.basename(path), 1, "\n".join(parts))

def load_pptx(path):
    from pptx import Presentation
    chunks = []
    for sno, slide in enumerate(Presentation(path).slides, 1):
        texts = [sh.text_frame.text for sh in slide.shapes if sh.has_text_frame and sh.text_frame.text.strip()]
        if texts:
            chunks += chunk_text(os.path.basename(path), sno, "\n".join(texts))
    return chunks

def load_any(path, describe_figures=True, say=print):
    ext = os.path.splitext(path)[1].lower()
    name = os.path.basename(path)
    if ext == ".pdf":
        return load_pdf(path, describe_figures, say)
    if ext == ".docx":
        return load_docx(path)
    if ext == ".pptx":
        return load_pptx(path)
    if ext in (".png", ".jpg", ".jpeg", ".webp"):
        say(f"📷 {name}: reading image")
        desc = vision(path, IMAGE_DOC_PROMPT, max_new_tokens=700)
        return [{"source": name, "page": 1, "kind": "figure", "image": path,
                 "text": f"[Image {name}] {desc}"}]
    if ext in (".txt", ".md", ".csv"):
        return chunk_text(name, 1, open(path, encoding="utf-8", errors="ignore").read())
    raise ValueError(f"unsupported file type '{ext}' (use PDF, Word, PowerPoint, TXT, MD, CSV or an image)")

def build_chunks(paths, describe_figures=True, errors=None, say=print):
    """Load files into chunks. Problems are collected in `errors` instead of being silently skipped."""
    chunks = []
    for p in paths:
        try:
            new = load_any(p, describe_figures, say)
            if not new:
                raise ValueError("no readable text found")
        except Exception as e:
            msg = f"{os.path.basename(p)}: {e}"
            print("⚠️ Skipping", msg)
            if errors is not None:
                errors.append(msg)
            continue
        for c in new:
            c["id"] = len(chunks)
            chunks.append(c)
    return chunks

DOC_PATHS = sorted(f"docs/{f}" for f in os.listdir("docs"))
if not DOC_PATHS:  # offline fallback so the notebook never breaks
    open("docs/fallback.txt", "w").write(
        "Retrieval-augmented generation (RAG) combines a retriever with a generator. "
        "The retriever finds relevant passages and the generator writes an answer conditioned on them.")
    DOC_PATHS = ["docs/fallback.txt"]

t0 = time.time()
CHUNKS = build_chunks(DOC_PATHS)
n_fig = sum(c["kind"] == "figure" for c in CHUNKS)
print(f"✅ {len(DOC_PATHS)} documents → {len(CHUNKS) - n_fig} text chunks + {n_fig} figure chunks in {time.time()-t0:.0f}s")

In [ ]:
# Peek at one figure: the rendered page and what the vision model saw
fig_chunk = next((c for c in CHUNKS if c["kind"] == "figure"), None)
if fig_chunk:
    from IPython.display import display
    im = Image.open(fig_chunk["image"]); im.thumbnail((600, 800)); display(im)
    print(fig_chunk["text"][:800], "...")

## 5 · Hybrid search index
- **BM25** finds exact keyword matches (names, acronyms, "Figure 2").
- **Dense vectors** find meaning matches (paraphrases).
- **Reciprocal Rank Fusion** merges both rankings.
- **Cross-encoder reranker** reads question + passage together for a sharper final ranking.

In [ ]:
import faiss
from rank_bm25 import BM25Okapi

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # bge query instruction
def tokenize(t):
    return re.findall(r"[a-z0-9]+", t.lower())

class HybridIndex:
    def __init__(self, chunks):
        self.chunks = chunks
        self.bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])
        emb = embedder.encode([c["text"] for c in chunks], batch_size=64,
                              normalize_embeddings=True, show_progress_bar=False)
        emb = np.asarray(emb, dtype="float32")
        self.faiss = faiss.IndexFlatIP(emb.shape[1])
        self.faiss.add(emb)

    def bm25_search(self, q, k=20):
        scores = self.bm25.get_scores(tokenize(q))
        return [int(i) for i in np.argsort(-scores)[:k]]

    def dense_search(self, q, k=20):
        e = np.asarray(embedder.encode([QUERY_PREFIX + q], normalize_embeddings=True), dtype="float32")
        _, I = self.faiss.search(e, min(k, len(self.chunks)))
        return [int(i) for i in I[0] if i != -1]

    def hybrid_search(self, q, k=20, rrf_k=60):
        scores = {}
        for ranked in (self.bm25_search(q, k), self.dense_search(q, k)):
            for rank, i in enumerate(ranked):
                scores[i] = scores.get(i, 0) + 1 / (rrf_k + rank + 1)
        return [i for i, _ in sorted(scores.items(), key=lambda x: -x[1])[:k]]

    def rerank(self, q, ids, k=TOP_K):
        if not ids:
            return []
        sc = reranker.predict([(q, self.chunks[i]["text"]) for i in ids])
        order = np.argsort(-sc)[:k]
        return [(ids[j], float(sc[j])) for j in order]

    def search(self, q, k=TOP_K, mode="hybrid+rerank"):
        if mode == "bm25":   return self.bm25_search(q, k)
        if mode == "dense":  return self.dense_search(q, k)
        if mode == "hybrid": return self.hybrid_search(q, k)
        return [i for i, _ in self.rerank(q, self.hybrid_search(q, 20), k)]

t0 = time.time()
INDEX = HybridIndex(CHUNKS)
print(f"✅ Index built in {time.time()-t0:.1f}s")
for i in INDEX.search("Transformer model architecture figure encoder decoder", k=3):
    c = INDEX.chunks[i]
    print(f"- [{c['kind']}] {c['source']} p.{c['page']}: {c['text'][:140]}...")

## 6 · The agent: plan → retrieve → grade → answer (with figure image) → verify
Each step is a small, focused LLM call, which makes a 3B model far more reliable than one giant prompt. Every decision is logged in a **trace**.

In [ ]:
def plan_queries(question):
    prompt = f"""Break the user's question into 1-3 short search queries that together cover everything needed to answer it.
Simple questions need only 1 query.
Return ONLY a JSON list of strings.

Question: {question}"""
    qs = parse_json(llm(prompt, max_new_tokens=120), [question])
    qs = [q for q in qs if isinstance(q, str) and q.strip()][:3] if isinstance(qs, list) else []
    return qs or [question]

def grade_passages(question, index, ids):
    """One yes/no call per passage — small models are far more reliable at this than picking from a list."""
    kept, votes = [], []
    for i in ids:
        prompt = f"""Question: {question}

Passage:
{index.chunks[i]['text'][:700]}

Does this passage contain information that helps answer the question? Answer with only "yes" or "no"."""
        v = llm(prompt, max_new_tokens=3).strip().lower()
        votes.append("yes" if v.startswith("yes") else "no")
        if v.startswith("yes"):
            kept.append(i)
    return kept, votes

ANSWER_SYSTEM = ("You answer questions using ONLY the provided sources. "
                 "Cite sources inline like [1] or [2][3] after each claim. "
                 "If the sources do not contain the answer, say so plainly. Be concise. "
                 "Use earlier conversation turns only to understand what the user means, never as a source of facts.")
GENERAL_SYSTEM = "You are a helpful, knowledgeable assistant. Answer clearly and concisely. If unsure, say so."
VISUAL_RE = re.compile(r"\b(show|image|images|picture|pic|figure|fig|diagram|chart|graph|plot|table|illustrat\w*|visual\w*|draw\w*|photo|look like|screenshot)\b", re.I)

def format_context(index, ids, with_meta=True):
    parts = []
    for n, i in enumerate(ids, 1):
        c = index.chunks[i]
        head = f"[{n}] ({c['kind']}, source: {c['source']}, page {c['page']})" if with_meta else f"[{n}]"
        parts.append(f"{head}\n{c['text']}")
    return "\n\n".join(parts)

def first_figure(index, ids):
    return next((index.chunks[i]["image"] for i in ids if index.chunks[i].get("image")), None)

def generate_answer(question, index, ids, strict=False, history=None):
    extra = "\nEvery sentence MUST be directly supported by a source. Remove anything you cannot cite." if strict else ""
    img_path = first_figure(index, ids)
    img = Image.open(img_path).convert("RGB") if img_path else None
    note = "\n(The attached image is the page containing the figure/table from the sources — use it.)" if img else ""
    prompt = f"Sources:\n{format_context(index, ids)}{note}\n\nQuestion: {question}{extra}\n\nAnswer with citations:"
    return llm(prompt, system=ANSWER_SYSTEM, max_new_tokens=350, image=img, history=history)

def verify_answer(answer, index, ids):
    prompt = f"""Sources:
{format_context(index, ids, with_meta=False)}

Answer to check:
{answer}

Is every claim in the answer supported by the sources?
Return ONLY JSON: {{"supported": true or false, "unsupported_claims": ["..."]}}"""
    v = parse_json(llm(prompt, max_new_tokens=150), None)
    return v if isinstance(v, dict) else {"supported": True, "unsupported_claims": [], "note": "verifier output unparseable"}

def ask(question, index, k=TOP_K, verbose=True, allow_general=False, history=None):
    t0 = time.time()
    trace = {}

    # 1. Plan
    queries = plan_queries(question)
    trace["1_planned_queries"] = queries

    # 2. Retrieve (hybrid for every sub-query) + rerank against the original question
    candidates = []
    for q in queries:
        for i in index.hybrid_search(q, 15):
            if i not in candidates:
                candidates.append(i)
    ranked = index.rerank(question, candidates, k)
    trace["2_retrieved"] = [{"chunk": i, "kind": index.chunks[i]["kind"], "source": index.chunks[i]["source"],
                             "page": index.chunks[i]["page"], "rerank_score": round(s, 2)} for i, s in ranked]
    ids = [i for i, _ in ranked]

    # 3. Grade (corrective RAG)
    kept, votes = grade_passages(question, index, ids)
    trace["3_grading"] = {"votes": votes}
    if not kept and ranked and ranked[0][1] > RERANK_CONFIDENT:
        kept = ids[:3]
        trace["3_grading"]["decision"] = "LLM rejected all passages, but reranker is very confident → keeping top 3"
    elif not kept:
        trace["3_grading"]["decision"] = "No relevant passages in the documents"
        if allow_general:
            ans = ("⚠️ *Not found in your documents — answering from general knowledge:*\n\n"
                   + llm(question, system=GENERAL_SYSTEM, history=history))
            trace["3_grading"]["decision"] += " → answered from general knowledge (labelled)"
        else:
            ans = "I couldn't find this in the provided documents."
            trace["3_grading"]["decision"] += " → refusing instead of hallucinating"
        result = {"question": question, "answer": ans, "sources": [], "figures": [],
                  "trace": trace, "latency_s": round(time.time() - t0, 1)}
        if verbose: show(result)
        return result
    else:
        trace["3_grading"]["decision"] = f"kept {len(kept)} of {len(ids)} passages"

    # 3b. Visual question ("show me the diagram…")? Make sure a figure is included; else show the source pages.
    previews = []
    if VISUAL_RE.search(question) and not any(index.chunks[i]["kind"] == "figure" for i in kept):
        fig_ids = [i for i in index.hybrid_search(question, 40) if index.chunks[i]["kind"] == "figure"]
        best = index.rerank(question, fig_ids, 1)
        kept_pages = {(index.chunks[i]["source"], index.chunks[i]["page"]) for i in kept}
        if best and (best[0][1] > -1 or (index.chunks[best[0][0]]["source"], index.chunks[best[0][0]]["page"]) in kept_pages):
            kept = [best[0][0]] + kept[:TOP_K - 1]
            trace["3b_visual"] = f"visual question → added figure (chunk {best[0][0]}, score {best[0][1]:.2f})"
        else:
            previews = [p for p in (page_preview(index.chunks[i]) for i in kept[:2]) if p]
            trace["3b_visual"] = "visual question, no matching figure → showing pages of the top sources"

    # 4. Generate (the figure image is passed to the model if a figure chunk was kept)
    answer = generate_answer(question, index, kept, history=history)

    # 5. Verify, and rewrite once if unsupported
    check = verify_answer(answer, index, kept)
    trace["4_verification"] = check
    if check.get("supported") is False:
        answer = generate_answer(question, index, kept, strict=True, history=history)
        trace["5_regenerated"] = True

    sources = [{"n": n, "source": index.chunks[i]["source"], "page": index.chunks[i]["page"],
                "kind": index.chunks[i]["kind"], "image": index.chunks[i]["image"],
                "text": index.chunks[i]["text"]} for n, i in enumerate(kept, 1)]
    result = {"question": question, "answer": answer, "sources": sources,
              "figures": list(dict.fromkeys([s["image"] for s in sources if s["image"]] + previews)),
              "trace": trace, "latency_s": round(time.time() - t0, 1)}
    if verbose: show(result)
    return result

def show(r):
    print("❓", r["question"])
    print("\n💬", r["answer"])
    for s in r["sources"]:
        print(f"   [{s['n']}] {'📷 ' if s['kind'] == 'figure' else ''}{s['source']} p.{s['page']}")
    print("\n🧭 Trace:", json.dumps(r["trace"], indent=2))
    print(f"⏱️ {r['latency_s']}s\n" + "─" * 80)

In [ ]:
_ = ask("What is multi-head attention and why is it useful?", INDEX)
_ = ask("What does the Transformer architecture diagram show about the encoder and decoder?", INDEX)  # should use a figure
_ = ask("What is the capital of Australia?", INDEX)   # out-of-scope → should refuse

## 7 · Conversation memory 💬
Real assistants handle follow-ups. Before searching, the agent:
1. **Routes** the message: small talk or questions about the chat itself ("what did I ask first?") are answered from memory; everything else goes to the documents.
2. **Rewrites follow-ups** into standalone questions: *"how many did they use?"* → *"How many attention heads did the Transformer use?"*. Search engines don't understand "they", so this step is essential.
3. **Saves** every turn to `chat_memory.json` (or your Google Drive, see section 2).
4. If you **attach an image**, the vision model answers about it and links it to related passages in your documents.

In [ ]:
class ChatMemory:
    def __init__(self, path=MEMORY_PATH):
        self.path = path
        try:
            self.turns = json.load(open(path)) if os.path.exists(path) else []
        except Exception:
            self.turns = []

    def add(self, user, assistant, standalone=None, image=None):
        self.turns.append({"user": user, "assistant": assistant, "standalone": standalone,
                           "image": image, "time": time.strftime("%Y-%m-%d %H:%M")})
        try:
            json.dump(self.turns, open(self.path, "w"), indent=1)
        except Exception as e:
            print("⚠️ could not save memory:", e)

    def history(self, n=HISTORY_TURNS, max_chars=600):
        """Last n turns as chat messages for the model (long answers trimmed to save memory)."""
        msgs = []
        for t in self.turns[-n:]:
            msgs.append({"role": "user", "content": t["user"] + (" [attached an image]" if t.get("image") else "")})
            msgs.append({"role": "assistant", "content": t["assistant"][:max_chars]})
        return msgs

    def clear(self):
        self.turns = []
        if os.path.exists(self.path):
            os.remove(self.path)

MEMORY = ChatMemory()
print(f"💾 Memory loaded: {len(MEMORY.turns)} earlier turns")

def route(message):
    prompt = f"""Classify the user's message.
Answer "chat" if it is a greeting, thanks, small talk, or a question about this conversation itself
(e.g. "what did I ask before?", "summarize our chat", "repeat your last answer").
Answer "docs" for any other question.

Message: {message}
Answer with only one word: chat or docs."""
    return "chat" if llm(prompt, max_new_tokens=3).strip().lower().startswith("chat") else "docs"

def condense(message, memory):
    """Turn a follow-up into a standalone question using recent history."""
    if not memory.turns:
        return message
    convo = "\n".join(f"User: {t['user']}\nAssistant: {t['assistant'][:300]}" for t in memory.turns[-HISTORY_TURNS:])
    prompt = f"""Conversation so far:
{convo}

Follow-up message: {message}

Rewrite the follow-up as a standalone question that makes sense without the conversation
(replace words like "it", "they", "that figure", "the second one" with what they refer to).
If it is already standalone, return it unchanged. Return ONLY the question."""
    out = llm(prompt, max_new_tokens=80).strip().strip('"').strip()
    return out if 3 < len(out) < 400 else message

CHAT_SYSTEM = ("You are DocuMind, a friendly assistant for questions about the user's documents. "
               "Answer small talk briefly. For questions about this conversation, use the chat history.")
IMAGE_SYSTEM = ("You are DocuMind. Look carefully at the user's image and answer their question about it. "
                "If related document passages are given, connect the image to them and cite them like [1].")

def chat(message, index, image_path=None, allow_general=True, memory=MEMORY):
    t0 = time.time()
    message = (message or "").strip()
    history = memory.history()

    # A) user attached an image → answer about the image, linked to related document passages
    if image_path:
        q = condense(message or "Describe this image in detail.", memory)
        img = Image.open(image_path).convert("RGB")
        ranked = index.rerank(q, index.hybrid_search(q, 15), 3) if message else []
        related = [i for i, s in ranked if s > 0]
        ctx = f"\n\nRelated passages from the documents:\n{format_context(index, related)}" if related else ""
        answer = llm(f"{q}{ctx}", system=IMAGE_SYSTEM, image=img, history=history, max_new_tokens=400)
        sources = [{"n": n, "source": index.chunks[i]["source"], "page": index.chunks[i]["page"],
                    "kind": index.chunks[i]["kind"], "image": index.chunks[i]["image"]} for n, i in enumerate(related, 1)]
        r = {"route": "image", "standalone": q, "answer": answer, "sources": sources,
             "figures": [image_path] + [s["image"] for s in sources if s["image"]],
             "trace": {"route": "image", "standalone_question": q, "related_passages": related}}

    # B) small talk / questions about the conversation → answer from memory
    elif route(message) == "chat":
        answer = llm(message, system=CHAT_SYSTEM, history=memory.history(n=8), max_new_tokens=300)
        r = {"route": "chat", "standalone": message, "answer": answer, "sources": [], "figures": [],
             "trace": {"route": "chat (answered from conversation memory)"}}

    # C) document question → rewrite follow-up, then run the full agent
    else:
        q = condense(message, memory)
        res = ask(q, index, verbose=False, allow_general=allow_general, history=history)
        r = {"route": "docs", "standalone": q, "answer": res["answer"], "sources": res["sources"],
             "figures": res["figures"], "trace": {"route": "docs", "standalone_question": q, **res["trace"]}}

    r["latency_s"] = round(time.time() - t0, 1)
    memory.add(message or "(image)", r["answer"], r["standalone"], image_path)
    return r

In [ ]:
# Demo: a multi-turn conversation with follow-ups
demo_memory = ChatMemory(path="demo_memory.json"); demo_memory.clear()
for m in ["What is multi-head attention?",
          "How many heads did they use in the base model?",       # "they" → needs memory
          "Is there a figure that illustrates it?",               # "it" → multi-head attention
          "What was my first question?"]:                          # answered from memory
    r = chat(m, INDEX, memory=demo_memory)
    print(f"👤 {m}")
    if r["standalone"] != m and r["route"] == "docs":
        print(f"   🔁 understood as: {r['standalone']}")
    print(f"🤖 ({r['route']}, {r['latency_s']}s) {r['answer'][:400]}")
    if r["figures"]:
        print("   📷", r["figures"])
    print("─" * 80)

## 8 · Chat app 🚀
Run this cell and open the **`https://xxxx.gradio.live`** link. It works on your phone too.
- Ask follow-ups naturally; the agent remembers the conversation.
- Figures used in an answer appear on the right.
- Attach an image (a screenshot of a chart, a diagram, a photo of notes) and ask about it.
- Upload your own PDFs to chat with them.

The link stays live while this Colab session is running. Record a 60-second demo of it for LinkedIn/GitHub.

In [ ]:
import gradio as gr

STATE = {"index": INDEX}

def memory_as_messages():
    msgs = []
    for t in MEMORY.turns:
        msgs.append({"role": "user", "content": t["user"] + (" 📎" if t.get("image") else "")})
        msgs.append({"role": "assistant", "content": t["assistant"]})
    return msgs

def make_chatbot():
    try:
        return gr.Chatbot(value=memory_as_messages(), type="messages", height=480, label="DocuMind")
    except TypeError:  # newer Gradio: messages format is the default and `type` was removed
        return gr.Chatbot(value=memory_as_messages(), height=480, label="DocuMind")

def format_reply(r, message):
    parts = []
    if r["route"] == "docs" and r["standalone"].strip().lower() != message.strip().lower():
        parts.append(f"_🔁 Understood as: {r['standalone']}_\n")
    parts.append(r["answer"])
    if r["sources"]:
        parts.append("\n**Sources**")
        for s in r["sources"]:
            parts.append(f"- [{s['n']}] {'📷 ' if s['kind'] == 'figure' else ''}{s['source']}, p.{s['page']}")
    parts.append(f"\n<sub>{r['route']} · {r['latency_s']}s</sub>")
    return "\n".join(parts)

def respond(message, image, allow_general, history):
    history = list(history or [])
    if not (message or "").strip() and not image:
        return history, "", None, [], "Type a question or attach an image."
    try:
        r = chat(message, STATE["index"], image_path=image, allow_general=allow_general)
        reply = format_reply(r, message or "")
        trace = "```json\n" + json.dumps(r["trace"], indent=2, default=str) + "\n```"
        figs = r["figures"]
    except Exception as e:
        reply, trace, figs = f"⚠️ Something went wrong: `{e}`", "", []
    history += [{"role": "user", "content": (message or "(image)") + (" 📎" if image else "")},
                {"role": "assistant", "content": reply}]
    return history, "", None, figs, trace

def build_from_upload(files, keep_samples, progress=gr.Progress()):
    if not files:
        return "⚠️ Upload at least one file first."
    paths = [f if isinstance(f, str) else (getattr(f, "path", None) or f.name) for f in files]
    errors = []

    def say(msg):  # live progress in the app (and in the Colab log)
        print(msg)
        m = re.search(r"\((\d+)/(\d+)\)", msg)
        progress(int(m.group(1)) / int(m.group(2)) if m else None, desc=msg)

    progress(0, desc="Reading files…")
    new = build_chunks(paths, errors=errors, say=say)
    if not new:
        return "⚠️ Nothing could be read:\n" + "\n".join(f"- {e}" for e in errors)

    chunks = ([dict(c) for c in CHUNKS] if keep_samples else []) + [dict(c) for c in new]
    for i, c in enumerate(chunks):
        c["id"] = i
    progress(1.0, desc="Building search index…")
    STATE["index"] = HybridIndex(chunks)

    lines = [f"✅ Ready! {len(chunks)} chunks indexed" + (" (your files + sample papers)" if keep_samples else "") + "."]
    for name in dict.fromkeys(c["source"] for c in new):
        n_txt = sum(c["source"] == name and c["kind"] == "text" for c in new)
        n_fig = sum(c["source"] == name and c["kind"] == "figure" for c in new)
        lines.append(f"- **{name}**: {n_txt} text chunks, {n_fig} figures/tables"
                     + ("" if n_fig else " (none detected; answers will show page previews instead)"))
    if errors:
        lines.append("\n⚠️ Skipped:\n" + "\n".join(f"- {e}" for e in errors))
    lines.append("\nTip: ask *\"show me the diagram of …\"* to get images in the answer.")
    return "\n".join(lines)

def clear_all():
    MEMORY.clear()
    return [], [], "", "🗑️ Chat and memory cleared."

with gr.Blocks(title="DocuMind — Multimodal Agentic RAG") as demo:
    gr.Markdown("# 🧠 DocuMind\nChat with your documents, figures included. Remembers the conversation · "
                "cites sources · 100% open-source. Loaded: *Attention Is All You Need* + the original *RAG* paper.")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = make_chatbot()
            with gr.Row():
                msg = gr.Textbox(placeholder="Ask anything… follow-ups welcome", show_label=False, scale=4)
                send = gr.Button("Send", variant="primary", scale=1)
            img_in = gr.Image(type="filepath", label="📎 Attach an image (optional)", height=160)
            with gr.Row():
                allow = gr.Checkbox(value=True, label="Use general knowledge when it's not in the documents (clearly labelled)")
                clear = gr.Button("🗑️ Clear chat & memory")
            gr.Examples(["What is multi-head attention?", "How many heads did they use?",
                         "Show me the figure of the Transformer architecture", "What was my first question?"], inputs=msg)
        with gr.Column(scale=2):
            gallery = gr.Gallery(label="📷 Figures / images used in the last answer", height=320, columns=2)
            with gr.Accordion("🧭 Agent trace (what the agent did)", open=False):
                trace_md = gr.Markdown()
            with gr.Accordion("📚 Use my own documents", open=False):
                files = gr.File(file_count="multiple",
                                file_types=[".pdf", ".docx", ".pptx", ".txt", ".md", ".csv", ".png", ".jpg", ".jpeg", ".webp"],
                                label="PDF / Word / PowerPoint / TXT / images (scanned PDFs OK)")
                keep = gr.Checkbox(value=False, label="Keep the sample papers too")
                build_btn = gr.Button("Build index from my files", variant="primary")
                gr.Markdown("<sub>PDFs with many figures take a few minutes, since each figure page is read by the vision model.</sub>")
                status = gr.Markdown()

    send.click(respond, [msg, img_in, allow, chatbot], [chatbot, msg, img_in, gallery, trace_md])
    msg.submit(respond, [msg, img_in, allow, chatbot], [chatbot, msg, img_in, gallery, trace_md])
    build_btn.click(build_from_upload, [files, keep], status)
    clear.click(clear_all, None, [chatbot, gallery, trace_md, status])

demo.launch(share=True)

## 9 · Evaluation (optional, ~15 min) — the part most portfolios skip
Set `RUN_EVAL = True` in the last cell and run it. It's off by default so it doesn't slow down your chat.

We compare **three pipelines** to find out *which part* of the system helps:

| Pipeline | Retrieval | Agent steps |
|---|---|---|
| **naive RAG** | dense vectors only (the typical tutorial) | none |
| **hybrid RAG** | BM25 + dense + rerank | none |
| **agentic RAG** | BM25 + dense + rerank, per sub-query | plan → grade → verify |

Tests: single-hop and multi-hop questions (generated from text chunks, reference lists skipped), retrieval hit@5 / MRR, LLM-judge faithfulness & correctness, multi-hop **context recall**, and a **refusal test** on questions the documents can't answer.

In [ ]:
def is_reference_chunk(t):
    """Bibliography chunks make useless test questions ('What is the title of the paper by…')."""
    years = len(re.findall(r"\b(?:19|20)\d{2}\b", t))
    cites = len(re.findall(r"et al\.|arXiv|Proceedings|In Advances|Journal|\bpp\.", t))
    return years >= 4 and cites >= 2

def make_single_hop(chunks, n=12):
    rng = random.Random(0)
    tests = []
    for c in rng.sample(chunks, min(n, len(chunks))):
        prompt = f"""Read this passage and write ONE specific question that can be answered only from it, plus the short answer.
Do not mention "the passage" or "the text". Return ONLY JSON: {{"question": "...", "answer": "..."}}

Passage:
{c['text']}"""
        out = parse_json(llm(prompt, max_new_tokens=150), None)
        if isinstance(out, dict) and out.get("question"):
            tests.append({"question": out["question"], "answer": str(out.get("answer", "")),
                          "gold_chunks": [c["id"]], "type": "single-hop"})
    return tests

def make_multi_hop(index, chunks, n=6):
    """Pair a chunk with a related chunk from a different page, and ask for a question needing both."""
    good_ids = {c["id"] for c in chunks}
    rng = random.Random(1)
    tests, used = [], set()
    for a in rng.sample(chunks, len(chunks)):
        if len(tests) >= n:
            break
        if a["id"] in used:
            continue
        partner = next((i for i in index.dense_search(a["text"][:500], 15)
                        if i in good_ids and i not in used and index.chunks[i]["page"] != a["page"]), None)
        if partner is None:
            continue
        b = index.chunks[partner]
        prompt = f"""Here are two passages.

Passage A:
{a['text']}

Passage B:
{b['text']}

Write ONE question that needs facts from BOTH passages to answer fully, plus a short answer that combines them.
Do not mention "passage A", "passage B" or "the text". Return ONLY JSON: {{"question": "...", "answer": "..."}}"""
        out = parse_json(llm(prompt, max_new_tokens=200), None)
        if isinstance(out, dict) and out.get("question"):
            tests.append({"question": out["question"], "answer": str(out.get("answer", "")),
                          "gold_chunks": [a["id"], b["id"]], "type": "multi-hop"})
            used.update([a["id"], b["id"]])
    return tests

def eval_retrieval(index, tests, k=TOP_K):
    rows = []
    for mode in ["bm25", "dense", "hybrid", "hybrid+rerank"]:
        hits, rr = 0, 0.0
        for t in tests:
            gold = t["gold_chunks"][0]
            res = index.search(t["question"], k, mode)
            if gold in res:
                hits += 1
                rr += 1 / (res.index(gold) + 1)
        rows.append({"retriever": mode, f"hit@{k}": round(hits / len(tests), 3), "MRR": round(rr / len(tests), 3)})
    return pd.DataFrame(rows)

def simple_rag(question, index, mode, k=TOP_K):
    """One search, top-k into the prompt, answer. No planning, grading or verification."""
    t0 = time.time()
    ids = index.search(question, k, mode=mode)
    ans = generate_answer(question, index, ids)
    return {"answer": ans, "retrieved": ids, "sources": [{"text": index.chunks[i]["text"]} for i in ids],
            "latency_s": round(time.time() - t0, 1)}

def agentic_rag(question, index):
    r = ask(question, index, verbose=False, allow_general=False)  # strict mode for a fair test
    r["retrieved"] = [d["chunk"] for d in r["trace"].get("2_retrieved", [])]
    return r

def judge(question, reference, answer, context):
    prompt = f"""You are a strict grader.
Question: {question}
Reference answer: {reference}
Retrieved context: {context[:2500]}
Model answer: {answer}

Score from 1 to 5:
- faithfulness: is the model answer supported by the retrieved context? (5 = fully supported)
- correctness: does it agree with the reference answer? (5 = fully correct)
Return ONLY JSON: {{"faithfulness": n, "correctness": n}}"""
    s = parse_json(llm(prompt, max_new_tokens=40), {})
    if not isinstance(s, dict):
        return None, None
    def num(x):
        try: return max(1, min(5, int(x)))
        except Exception: return None
    return num(s.get("faithfulness")), num(s.get("correctness"))

OUT_OF_SCOPE = [
    "What is the capital of Australia?",
    "Who won the 2022 FIFA World Cup?",
    "What is a good recipe for butter chicken?",
    "How many parameters does GPT-4 have?",
    "How does LoRA fine-tuning reduce memory usage?",
    "How is RLHF used to train ChatGPT?",
]
REFUSAL_RE = re.compile(
    r"couldn.?t find|could not find|no information|unable to|cannot (?:be )?(?:answer|determine|find)|"
    r"(?:does|do) not (?:contain|mention|provide|include|say|discuss|cover)|"
    r"not (?:contain|include|mention|provide|cover|discuss)|not (?:available|present|found|mentioned) in", re.I)

In [ ]:
def run_evaluation(index):
    import matplotlib.pyplot as plt
    from IPython.display import display

    text_chunks = [c for c in index.chunks if c["kind"] == "text" and len(c["text"]) > 400
                   and not is_reference_chunk(c["text"])]
    print(f"{len(text_chunks)} usable text chunks for test questions")
    t0 = time.time()
    single, multi = make_single_hop(text_chunks, 12), make_multi_hop(index, text_chunks, 6)
    print(f"✅ {len(single)} single-hop + {len(multi)} multi-hop questions in {time.time()-t0:.0f}s")

    ret_df = eval_retrieval(index, single)
    display(ret_df)

    pipelines = {"naive RAG": lambda q: simple_rag(q, index, "dense"),
                 "hybrid RAG": lambda q: simple_rag(q, index, "hybrid+rerank"),
                 "agentic RAG": lambda q: agentic_rag(q, index)}
    rows = []
    for t in single[:6] + multi[:6]:
        for name, fn in pipelines.items():
            r = fn(t["question"])
            f, c = judge(t["question"], t["answer"], r["answer"], "\n".join(s["text"] for s in r["sources"]))
            recall = sum(g in r["retrieved"] for g in t["gold_chunks"]) / len(t["gold_chunks"])
            rows.append({"pipeline": name, "type": t["type"], "faithfulness": f, "correctness": c,
                         "context_recall": recall, "latency_s": r["latency_s"]})
        print(f"done [{t['type']}]: {t['question'][:70]}")
    gen_raw = pd.DataFrame(rows)
    gen_df = gen_raw.groupby("pipeline", sort=False).mean(numeric_only=True).round(2).reset_index()
    by_type = gen_raw.pivot_table(index="pipeline", columns="type", values=["correctness", "context_recall"],
                                  aggfunc="mean", sort=False).round(2)

    ref_rows = []
    for name, fn in pipelines.items():
        refused = 0
        for q in OUT_OF_SCOPE:
            a = fn(q)["answer"]
            ok = bool(REFUSAL_RE.search(a)); refused += ok
            print(f"[{name}] {'🛑 refused ' if ok else '⚠️ answered'} | {q} → {a[:80].replace(chr(10), ' ')}")
        ref_rows.append({"pipeline": name, "refusal_rate": round(refused / len(OUT_OF_SCOPE), 2)})
    ref_df = pd.DataFrame(ref_rows)
    summary = gen_df.merge(ref_df, on="pipeline")
    display(summary); display(by_type)

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
    ret_df.set_index("retriever")[[f"hit@{TOP_K}", "MRR"]].plot.bar(ax=axes[0], rot=0, ylim=(0, 1),
                                                                   title="Retrieval quality (higher is better)")
    summary.set_index("pipeline")[["faithfulness", "correctness"]].plot.bar(ax=axes[1], rot=0, ylim=(0, 5),
                                                                            title="Answer quality, LLM-judge 1–5")
    mh = gen_raw[gen_raw["type"] == "multi-hop"].groupby("pipeline", sort=False)["context_recall"].mean()
    pd.DataFrame({"multi-hop context recall": mh,
                  "refusal rate (out-of-scope)": ref_df.set_index("pipeline")["refusal_rate"]}
                 ).plot.bar(ax=axes[2], rot=0, ylim=(0, 1), title="Multi-hop recall & safe refusals")
    for ax in axes:
        ax.grid(axis="y", alpha=0.3); ax.set_xlabel("")
    plt.tight_layout(); plt.savefig("eval_results.png", dpi=150); plt.show()

    json.dump({"model": LLM_ID, "embedder": EMBED_ID, "reranker": RERANK_ID,
               "n_single_hop": len(single), "n_multi_hop": len(multi), "n_out_of_scope": len(OUT_OF_SCOPE),
               "retrieval": ret_df.to_dict("records"), "summary": summary.to_dict("records"),
               "by_type": json.loads(by_type.to_json())}, open("results.json", "w"), indent=2)
    try:
        print(ret_df.to_markdown(index=False)); print(); print(summary.to_markdown(index=False)); print(); print(by_type.to_markdown())
    except Exception:
        print(ret_df); print(summary); print(by_type)

RUN_EVAL = False   # ← set to True and run this cell for the benchmark (~15 min)
if RUN_EVAL:
    run_evaluation(INDEX)
    # from google.colab import files; files.download("eval_results.png"); files.download("results.json")
else:
    print("Evaluation skipped. Set RUN_EVAL = True above and run this cell to benchmark.")

## ✅ Done — what next?
- **File → Save a copy in Drive** so your work isn't lost.
- For memory that survives disconnects, uncomment the Google Drive lines in section 2.
- When finished: **Runtime → Disconnect and delete runtime** to save your free GPU quota.